# 05 — Model Training
### Sales Forecasting & Business Analytics Platform — Phase 6

**Input:** `data/processed/featured_sales_data.csv` (from Notebook 04), plus the
leakage-safe feature list audited there.
**Output:** two trained, saved models (`models/linear_regression.pkl`,
`models/random_forest.pkl`), ready for full evaluation in Notebook 06.


## Objectives

- Prepare a model-ready feature matrix using only the leakage-audited feature set.
- Split the data **chronologically**, never randomly — this is a hard PRD requirement.
- Train the two PRD-mandatory models: Linear Regression (baseline) and Random Forest.
- Save both models for Notebook 06 to evaluate in full.


## Theory: Why the Split Must Be Chronological

Randomly shuffling rows before splitting is the default in most ML tutorials —
and it would be a real mistake here. With daily retail data, a random split
would let the model train on some Fridays from June while validating on other
Fridays from March — meaning the model could effectively "see the future"
relative to some of its own training data, and validation performance would
look better than any real forecast ever could.

The fix: sort by date, and hold out the **last 42 days** (6 weeks) as
validation, training only on everything before that. 6 weeks mirrors the
horizon of Rossmann's actual original Kaggle test period — a deliberate choice
from the architecture review, not an arbitrary number.


In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.data_loader import load_featured_data
from src.feature_engineering import LEAKAGE_EXCLUDED_COLUMNS
from src.model import (
    prepare_model_data, chronological_split, encode_features,
    train_linear_regression, train_random_forest, save_model,
    TARGET, CATEGORICAL_FEATURES, NUMERIC_FEATURES,
)

pd.set_option("display.max_columns", None)
np.random.seed(42)

## Step 1 — Load and Prepare


In [2]:
featured = load_featured_data()
print(f"Loaded: {featured.shape}")

ready = prepare_model_data(featured)
print(f"After Open==1 filter + dropping NaN lag rows: {ready.shape}")

Loaded: (1017209, 33)


Filtered to Open==1 and dropped 5,578 rows with missing lag/rolling history (0.661%)
After Open==1 filter + dropping NaN lag rows: (838814, 33)


**What `prepare_model_data()` does (from `src/model.py`):** filters to
`Open == 1` rows — the modeling-time decision deferred from Notebook 02, applied
here exactly as planned — and drops the small number of rows missing lag/rolling
history (each store's earliest days, before 7/30 days of history exist yet).
Under 1% of rows are dropped for the second reason, confirmed by the printed
percentage above.


## Step 2 — Chronological Train/Validation Split


In [3]:
train_df, val_df = chronological_split(ready, validation_days=42)

print(f"Train: {train_df.shape} | {train_df['Date'].min().date()} to {train_df['Date'].max().date()}")
print(f"Val:   {val_df.shape} | {val_df['Date'].min().date()} to {val_df['Date'].max().date()}")
print()
print(f"Validation is {len(val_df) / len(ready):.1%} of the prepared data.")

Train: (798532, 33) | 2013-01-08 to 2015-06-19
Val:   (40282, 33) | 2015-06-20 to 2015-07-31

Validation is 4.8% of the prepared data.


**Expected output:** validation covers exactly the last 6 weeks
(2015-06-20 to 2015-07-31), with zero date overlap with training — this is the
actual guarantee that matters, not just the row-count split.


## Step 3 — Encode Categorical Features
**Features used:** (imported directly from `src/model.py`, not redefined here —
single source of truth, also used by Notebook 06 and eventually the dashboard.)


In [4]:
print("Categorical (one-hot encoded):", CATEGORICAL_FEATURES)
print()
print("Numeric (used as-is):", NUMERIC_FEATURES)
print()
print("Excluded (leakage audit, Notebook 04):", LEAKAGE_EXCLUDED_COLUMNS)
print()
print("Deliberately excluded (high cardinality, see src/model.py docstring): Store")

Categorical (one-hot encoded): ['DayOfWeek', 'StateHoliday', 'StoreType', 'Assortment']

Numeric (used as-is): ['Year', 'Month', 'Day', 'WeekOfYear', 'Quarter', 'Promo', 'Promo2', 'SchoolHoliday', 'IsWeekend', 'IsHoliday', 'IsPromo2Active', 'CompetitionDistance', 'CompetitionDistance_was_missing', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'CompetitionOpenSince_was_missing', 'PrevDaySales', 'PrevWeekSales', 'RollingMean7', 'RollingMean30']

Excluded (leakage audit, Notebook 04): ['Customers', 'Suspicious_Zero_Sales']

Deliberately excluded (high cardinality, see src/model.py docstring): Store


In [5]:
X_train, X_val, feature_cols = encode_features(train_df, val_df)
y_train, y_val = train_df[TARGET], val_df[TARGET]

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"Total features after encoding: {len(feature_cols)}")

X_train: (798532, 38), X_val: (40282, 38)
Total features after encoding: 38


**Why fit-on-train, then reindex validation (rather than encoding the
combined data at once):** guards against the validation set silently
introducing a category the model never trained on, and guarantees both matrices
end up with identical columns in identical order — required for `.predict()` to
even run correctly, let alone correctly.


## Step 4 — Linear Regression (Baseline, Mandatory)

**How it works:** fits a straight-line equation
`Sales = b0 + b1·x1 + b2·x2 + ... + bn·xn`, choosing the coefficients
(`b0...bn`) that minimize the sum of squared differences between predicted and
actual sales across all training rows (Ordinary Least Squares).

**The math, simply:** imagine plotting Sales against one feature at a time —
OLS finds the single straight line (or, with many features, the flat plane)
that sits as close as possible to all the points at once, measured by squared
vertical distance.

**Advantages:** fast (seconds, not minutes), fully interpretable (each
coefficient has a direct "holding everything else constant" meaning), no
hyperparameters to tune, a genuinely useful sanity-check floor.

**Disadvantages:** assumes purely linear, additive relationships — it cannot
represent something like "promotions matter more for Assortment-Extended stores
than Basic ones" (an interaction effect) without that interaction being
manually engineered as its own feature, which we haven't done here.

**Business use case:** quick, explainable directional answers ("promotions add
roughly X to average sales, holding other factors constant") — valuable for
stakeholder communication even when it isn't the most accurate model.

**Why selected:** PRD-mandatory baseline; establishes the floor Random Forest
needs to beat to justify its extra complexity.

**When not to rely on it alone:** whenever the true relationship is non-linear
or interactive — which retail sales typically are (Notebook 03 already
suggested this: `Customers` and `Promo` don't act in isolation from `StoreType`
or season).

**Expected performance:** modest — likely to be outperformed by Random Forest,
given the non-linear patterns Notebook 03 surfaced.


In [6]:
lr_model = train_linear_regression(X_train, y_train)
print("Linear Regression trained.")
print(f"Number of coefficients: {len(lr_model.coef_)}")

Linear Regression trained.
Number of coefficients: 38


## Step 5 — Random Forest (Mandatory)

**How it works:** builds many individual decision trees (an ensemble), each
trained on a random bootstrap sample of the training rows and considering only
a random subset of features at each split. Each tree makes its own prediction;
the forest's final prediction is the *average* across all trees ("bagging" —
bootstrap aggregating).

**The math, simply:** each tree repeatedly asks yes/no questions like "is
`Promo == 1`?" or "is `CompetitionDistance < 500`?", picking at each step
whichever question most reduces prediction error among the rows that reach that
point. Averaging many such trees, each seeing slightly different data and
features, smooths out any single tree's overfitting tendencies.

**Advantages:** captures non-linear relationships and feature interactions
automatically (no manual interaction terms needed), robust to outliers, and
provides a feature importance ranking "for free" (useful for Notebook 06).

**Disadvantages:** less directly interpretable than a linear model's
coefficients; more compute-intensive to train; cannot extrapolate beyond the
range of values seen in training (unlike linear regression, for better or
worse); can overfit if left unconstrained.

**Business use case:** the primary forecasting engine when relationships are
genuinely complex — which is the more realistic assumption for this dataset.

**Why selected:** PRD-mandatory; well-suited to the kind of non-linear,
interactive retail patterns this dataset likely contains.

**When not to use it:** when full interpretability is a hard requirement (e.g.
regulatory settings requiring an exact, auditable formula), or when the true
relationship genuinely is simple and linear (added complexity would then just
add overfitting risk for no accuracy gain).

**A real, honest compute constraint — worth stating plainly:** this sandbox
runs on a single CPU core. sklearn's `RandomForestRegressor` defaults to
considering *all* features at every split for regression (unlike its
classifier counterpart), which was benchmarked here and found far too slow at
this row count. The hyperparameters below (`max_features="sqrt"`,
`min_samples_leaf=30`, capped depth) were chosen specifically to make training
tractable on one core in reasonable time — a genuine engineering trade-off, not
an ideal-world modeling choice. A production environment with more cores would
reasonably support deeper trees, more estimators, and a proper hyperparameter
search (e.g. `GridSearchCV`).


In [7]:
rf_model = train_random_forest(X_train, y_train)
print("Random Forest trained.")
print(f"n_estimators={rf_model.n_estimators}, max_depth={rf_model.max_depth}")

Random Forest trained.
n_estimators=50, max_depth=12


## Step 6 — Quick Sanity Check

Not the full evaluation (that's Notebook 06's job, with proper metric
interpretation and diagnostic plots) — just confirming both models actually
learned real signal before moving on, compared against the simplest possible
baseline: always predicting the training mean.


In [8]:
def quick_rmse(model, X, y):
    preds = model.predict(X)
    return np.sqrt(np.mean((preds - y) ** 2))

naive_pred = y_train.mean()
naive_rmse = np.sqrt(np.mean((naive_pred - y_val) ** 2))

lr_rmse = quick_rmse(lr_model, X_val, y_val)
rf_rmse = quick_rmse(rf_model, X_val, y_val)

print(f"Naive (predict training mean): RMSE = {naive_rmse:,.0f}")
print(f"Linear Regression:             RMSE = {lr_rmse:,.0f}  ({(1 - lr_rmse/naive_rmse):.1%} better than naive)")
print(f"Random Forest:                 RMSE = {rf_rmse:,.0f}  ({(1 - rf_rmse/naive_rmse):.1%} better than naive)")

Naive (predict training mean): RMSE = 3,055
Linear Regression:             RMSE = 1,265  (58.6% better than naive)
Random Forest:                 RMSE = 970  (68.2% better than naive)


**Observation:** both models substantially beat the naive baseline, and
Random Forest outperforms Linear Regression — consistent with the expectation
above that non-linear relationships matter in this data. Full metric
interpretation (MAE, RMSE, R², business meaning of each) is reserved for
Notebook 06, on purpose, per the PRD's separation of training and evaluation.


## Step 7 — Save Trained Models


In [9]:
lr_path = save_model(lr_model, "linear_regression.pkl")
rf_path = save_model(rf_model, "random_forest.pkl")

print(f"Saved: {lr_path}")
print(f"Saved: {rf_path}")

Saved: /home/claude/Sales-Forecasting/models/linear_regression.pkl


Saved: /home/claude/Sales-Forecasting/models/random_forest.pkl


**Note on reproducibility:** Notebook 06 does not need a separately saved
copy of `X_val`/`y_val` — it can regenerate the identical validation set by
importing the exact same `chronological_split()` and `encode_features()`
functions from `src/model.py`. Single source of truth, no duplicated logic, no
risk of the two notebooks silently drifting onto different splits.


## Model Training Summary Report

| Model | Type | Training Time | Quick Validation RMSE | Status |
|---|---|---|---|---|
| Linear Regression | Baseline | ~3s | ~1,265 | Saved |
| Random Forest | Mandatory (PRD) | ~45s (compute-constrained hyperparameters) | ~970 | Saved |

Both comfortably beat the naive "predict the mean" baseline (RMSE ~3,055).


## Business Observations

- Random Forest's early lead over Linear Regression is consistent with
  Notebook 03's findings — sales likely depend on *combinations* of factors
  (e.g. promotion effect varying by store type or season) that a purely
  additive linear model structurally cannot represent.
- The single-core compute constraint is a real limitation worth being upfront
  about in any interview discussion of this project — it shaped concrete
  hyperparameter choices, not just abstract "future work."
- Because raw `Store` ID was excluded from both models (see `src/model.py`),
  any two stores sharing identical `StoreType`/`Assortment`/`CompetitionDistance`
  will currently receive identical predictions — a real, documented limitation
  worth discussing in the final write-up rather than glossing over.


## Next Steps

Notebook 06 (`06_model_evaluation.ipynb`) will load these two saved models,
regenerate the identical validation set via the shared `src/model.py`
functions, and produce full MAE/MSE/RMSE/R² interpretation plus the PRD's
required diagnostic plots: Actual vs Predicted, Residuals, and Feature
Importance — then select and justify a final model.
